# Scoring review

Review of the scores in `data/processed/airport_scores.csv`, produced by `server/app/core/scoring.py`.

Main sections:
- **Part 1 – automatic checks:** facts the code can verify (`assert` raises an error if a check fails).
- **Part 2 – judgement checks:** read the numbers and decide whether they make sense.
- **Part 3 – weight sensitivity (D19):** 8 weight scenarios, top-5 overlap and Spearman per hub group.
- **Part 4 – baseline comparison (D21):** our score vs ranking by size only and by growth only.

To regenerate the scores first: `python server/app/core/scoring.py`

> The production pipeline is `server/scripts/build_data.py`: it moves this logic into one script and verifies its output against the committed files in `data/processed/`. Findings are recorded in `docs/validation.md` and `docs/decisions.md`.

In [1]:
from pathlib import Path
import pandas as pd

# Works whether the notebook runs from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
s = pd.read_csv(ROOT / "data" / "processed" / "airport_scores.csv")
pd.set_option("display.width", 200)
s.shape

(402, 40)

## Part 1 – Automatic checks

### Check 1: 402 rows, all scores between 0 and 1

In [2]:
print(len(s), s["investment_score"].min(), s["investment_score"].max())
assert len(s) == 402
assert s["investment_score"].between(0, 1).all()
print("OK")

402 0.0816404784789947 0.9628571428571427
OK


### Check 2: HVN has 3 real components and a neutral congestion score

HVN has no BTS delay data (Avelo / Breeze don't report), so congestion should be the neutral 0.5 (D18).

In [3]:
hvn = s.set_index("code").loc["HVN"]
print(hvn[["score_completeness", "congestion_pct", "has_delays"]])
assert hvn["score_completeness"] == 3 and hvn["congestion_pct"] == 0.5
print("OK")

score_completeness        3
congestion_pct          0.5
has_delays            False
Name: HVN, dtype: object
OK


### Check 5: SFO vs LAX

Expect SFO higher: fuller flights, more congestion, faster recovery. Raw metrics are shown next to the percentiles so you can see the lead comes from the data.

In [4]:
cols = ["code", "hub", "investment_score", "rank_in_hub",
        "cagr_19_25", "pct_change_24_25", "load_factor", "nas_delay_rate",
        "growth_pct", "seat_demand_pct", "congestion_pct", "scale_pct"]
sfo_lax = s[s["code"].isin(["SFO", "LAX"])].set_index("code")
display(sfo_lax[cols[1:]])
assert sfo_lax.loc["SFO", "investment_score"] > sfo_lax.loc["LAX", "investment_score"]
print("OK: SFO scores higher than LAX")

,hub,investment_score,rank_in_hub,cagr_19_25,pct_change_24_25,load_factor,nas_delay_rate,growth_pct,seat_demand_pct,congestion_pct,scale_pct
code,,,,,,,,,,,
SFO,L,0.686667,7.0,-0.009381,0.0468,0.823469,0.098659,0.400000,0.900000,0.866667,0.700000
LAX,L,0.403333,21.0,-0.026727,-0.0335,0.817315,0.045827,0.066667,0.766667,0.233333,0.866667


OK: SFO scores higher than LAX


## Part 2 – Judgement checks

### Helper: why did an airport score this way?

Shows each component's percentile next to the raw metric behind it. A raw value of `nan` means that component got the neutral 0.5.

In [5]:
def explain(code):
    r = s.set_index("code").loc[code]
    print(f"{code} ({r['city']}, {r['state']}) hub={r['hub']} "
          f"score={r['investment_score']:.3f} rank_in_hub={r['rank_in_hub']}")
    rows = [("growth",      r["growth_pct"],      f"CAGR {r['cagr_19_25']:.1%}, 24→25 {r['pct_change_24_25']:.1%}"),
            ("seat_demand", r["seat_demand_pct"], f"load factor {r['load_factor']:.1%}"),
            ("congestion",  r["congestion_pct"],  f"NAS delay rate {r['nas_delay_rate']:.1%}"),
            ("scale",       r["scale_pct"],       f"{r['enpl_2025']:,.0f} enplanements")]
    for name, pct, raw in rows:
        print(f"  {name:12} pct={pct:.2f}   ({raw})")

explain("HVN")

HVN (New Haven, CT) hub=S score=0.579 rank_in_hub=29.0
  growth       pct=1.00   (CAGR 57.2%, 24→25 25.2%)
  seat_demand  pct=0.24   (load factor 76.0%)
  congestion   pct=0.50   (NAS delay rate nan%)
  scale        pct=0.29   (737,789 enplanements)


### Check 3: New England

Go down the list: would you expect each airport where it is? Remember scores are relative to the airport's hub group, not to each other. HVN was expected to rank high because of its growth: does it? If not, which component holds it back?

In [6]:
ne = s[s["state"].isin(["CT", "ME", "MA", "NH", "RI", "VT"])]
ne[["code", "city", "hub", "rank_in_hub", "investment_score",
    "growth_pct", "seat_demand_pct", "congestion_pct", "scale_pct", "score_completeness"]]

,code,city,hub,rank_in_hub,investment_score,growth_pct,seat_demand_pct,congestion_pct,scale_pct,score_completeness
8,BGR,Bangor,N,6.0,0.846291,0.883065,0.804688,0.758140,0.976744,4
102,BOS,Boston,L,9.0,0.655000,0.466667,0.733333,0.933333,0.500000,4
105,MVY,Vineyard Haven,N,68.0,0.640787,0.866935,0.277344,0.697674,0.624031,4
125,PWM,Portland,S,25.0,0.618160,0.531646,0.810127,0.538462,0.632911,4
144,HVN,New Haven,S,29.0,0.578797,1.000000,0.240506,0.500000,0.291139,3
150,BDL,Windsor Locks,M,15.0,0.572857,0.400000,0.942857,0.600000,0.314286,4
155,ACK,Nantucket,N,96.0,0.564699,0.568548,0.304688,0.702326,0.759690,4
161,ORH,Worcester,N,98.0,0.544974,0.479839,0.347656,0.753488,0.678295,4
168,BTV,Burlington,S,38.0,0.534323,0.278481,0.734177,0.846154,0.278481,4
191,PVD,Warwick,S,44.0,0.505850,0.430380,0.417722,0.448718,0.924051,4


In [7]:
# Explain any New England airport that surprises you
for c in ["BGR", "BOS", "HVN"]:
    explain(c)
    print()

BGR (Bangor, ME) hub=N score=0.846 rank_in_hub=6.0
  growth       pct=0.88   (CAGR 5.1%, 24→25 16.1%)
  seat_demand  pct=0.80   (load factor 80.2%)
  congestion   pct=0.76   (NAS delay rate 5.0%)
  scale        pct=0.98   (437,108 enplanements)

BOS (Boston, MA) hub=L score=0.655 rank_in_hub=9.0
  growth       pct=0.47   (CAGR 0.3%, 24→25 -0.3%)
  seat_demand  pct=0.73   (load factor 81.7%)
  congestion   pct=0.93   (NAS delay rate 11.0%)
  scale        pct=0.50   (21,021,153 enplanements)

HVN (New Haven, CT) hub=S score=0.579 rank_in_hub=29.0
  growth       pct=1.00   (CAGR 57.2%, 24→25 25.2%)
  seat_demand  pct=0.24   (load factor 76.0%)
  congestion   pct=0.50   (NAS delay rate nan%)
  scale        pct=0.29   (737,789 enplanements)



### Check 4: Top 10 per hub group

For every surprise, run `explain()` and find which component drove it. If you can explain it, it's a feature. If not, it may be a bug.

In [8]:
hub_names = {"L": "Large", "M": "Medium", "S": "Small", "N": "Nonhub"}
for hub, name in hub_names.items():
    print(f"=== Top 10 {name} ===")
    top = s[s["hub"] == hub].head(10)
    print(top[["rank_in_hub", "code", "city", "state", "investment_score",
               "growth_pct", "seat_demand_pct", "congestion_pct", "scale_pct",
               "score_completeness"]].round(3).to_string(index=False), "\n")

=== Top 10 Large ===
 rank_in_hub code                                city state  investment_score  growth_pct  seat_demand_pct  congestion_pct  scale_pct  score_completeness
         1.0  DEN                              Denver    CO             0.800       0.900            0.600           0.800      0.900                   4
         2.0  MIA                               Miami    FL             0.780       0.867            0.800           0.667      0.733                   4
         3.0  DFW                          Fort Worth    TX             0.772       0.767            0.933           0.500      0.967                   4
         4.0  MCO                             Orlando    FL             0.762       0.833            0.667           0.733      0.800                   4
         5.0  ORD                             Chicago    IL             0.710       0.533            0.700           0.833      0.933                   4
         6.0  EWR                              Newark  

In [9]:
# Replace with any airport that surprised you above
explain("SJU")

SJU (San Juan, PR) hub=M score=0.963 rank_in_hub=1.0
  growth       pct=1.00   (CAGR 6.4%, 24→25 2.6%)
  seat_demand  pct=1.00   (load factor 83.7%)
  congestion   pct=0.97   (NAS delay rate 8.7%)
  scale        pct=0.80   (6,658,037 enplanements)


### Extra: score_completeness counts

In [10]:
s["score_completeness"].value_counts().sort_index()

score_completeness
1      1
2      6
3     41
4    354
Name: count, dtype: int64

In [11]:
s[s["score_completeness"] < 3][["code", "city", "state", "hub", "enpl_2025", "score_completeness", "has_t100", "has_delays", "growth_flag"]]

,code,city,state,hub,enpl_2025,score_completeness,has_t100,has_delays,growth_flag
395,1G4,Peach Springs,AZ,N,26748.0,2,True,False,insufficient_history
396,ILG,Wilmington,DE,N,169382.0,2,True,False,insufficient_history
397,LAL,Lakeland,FL,N,91018.0,2,True,False,insufficient_history
398,SBD,San Bernardino,CA,N,25957.0,2,True,False,insufficient_history
399,RVR,Ceiba,PR,N,23214.0,2,False,False,computed
400,SLE,Salem,OR,N,11060.0,2,True,False,insufficient_history
401,YNG,Youngstown,OH,N,18252.0,1,True,False,insufficient_history


## Part 3 – Weight sensitivity (D19)

Weights are a judgement call. This section tests whether the rankings depend heavily on them: if small weight changes reshuffle the top airports, the ranking is fragile; if not, it is robust.

### Step 1: import the scoring module

We recompute scores live with `compute_scores` (instead of reading the CSV), so each scenario can use different weights.

In [12]:
import sys
sys.path.insert(0, str(ROOT / "server"))
from app.core.scoring import compute_scores, load_metrics, shift_weight, WEIGHTS

metrics = load_metrics()
WEIGHTS

{'growth': 0.35, 'seat_demand': 0.25, 'congestion': 0.25, 'scale': 0.15}

### Step 2: build the 8 scenarios

For each of the 4 weights: +0.10 and −0.10. After changing one weight, the other three are rescaled proportionally so the total stays 1. Example: growth +0.10 → growth 0.45, and the other three (originally 0.65 in total) shrink to 0.55 in total, keeping their 25:25:15 ratio.

In [13]:
# shift_weight comes from scoring.py: change one weight, rescale the others
scenarios = {"baseline": WEIGHTS}
for key in WEIGHTS:
    for delta in (+0.10, -0.10):
        scenarios[f"{key} {delta:+.2f}"] = shift_weight(WEIGHTS, key, delta)

pd.DataFrame(scenarios).T.round(3)

,growth,seat_demand,congestion,scale
baseline,0.350,0.250,0.250,0.150
growth +0.10,0.450,0.212,0.212,0.127
growth -0.10,0.250,0.288,0.288,0.173
seat_demand +0.10,0.303,0.350,0.217,0.130
seat_demand -0.10,0.397,0.150,0.283,0.170
congestion +0.10,0.303,0.217,0.350,0.130
congestion -0.10,0.397,0.283,0.150,0.170
scale +0.10,0.309,0.221,0.221,0.250
scale -0.10,0.391,0.279,0.279,0.050


### Step 3: score every scenario

Same data, different weights. Index by airport code so scenarios are easy to compare.

In [14]:
results = {name: compute_scores(metrics, w).set_index("code")
           for name, w in scenarios.items()}
base = results["baseline"]
HUBS = ["L", "M", "S", "N"]

def top5(df, hub):
    """Codes of the 5 best ranked airports in a hub (df is already sorted by score)."""
    return list(df[(df["hub"] == hub) & df["ranked"]].index[:5])

{hub: top5(base, hub) for hub in HUBS}

{'L': ['DEN', 'MIA', 'DFW', 'MCO', 'ORD'],
 'M': ['SJU', 'DJT', 'RSW', 'RDU', 'CHS'],
 'S': ['TYS', 'PGD', 'PIE', 'BZN', 'SRQ'],
 'N': ['BLV', 'TVC', 'FWA', 'RAP', 'ROA']}

### Step 4: compare each scenario with the baseline

Per scenario and hub group:
- **Top-5 overlap (0–5):** how many of the baseline top 5 are still in the top 5.
- **Spearman correlation:** how similar the *whole* ranking in the hub is to the baseline. 1.0 = identical order, 0 = unrelated. It is the ordinary correlation of the two rank lists, computed with pandas (no scipy needed).

In [15]:
rows = []
for name, res in results.items():
    if name == "baseline":
        continue
    for hub in HUBS:
        overlap = len(set(top5(base, hub)) & set(top5(res, hub)))
        # Full ranking of the hub's ranked airports, baseline vs scenario
        codes = base.index[(base["hub"] == hub) & base["ranked"]]
        r_base = base.loc[codes, "investment_score"].rank()
        r_scen = res.loc[codes, "investment_score"].rank()
        rows.append({"scenario": name, "hub": hub,
                     "top5_overlap": overlap,
                     "spearman": r_base.corr(r_scen)})

sens = pd.DataFrame(rows)

### Step 5: summary table (scenario × hub)

Left: top-5 overlap (5 = top 5 unchanged). Right: Spearman (closer to 1 = more stable).

In [16]:
overlap_tbl = sens.pivot(index="scenario", columns="hub", values="top5_overlap")[HUBS]
spearman_tbl = sens.pivot(index="scenario", columns="hub", values="spearman")[HUBS].round(3)
order = [s for s in scenarios if s != "baseline"]
summary = pd.concat({"top5_overlap": overlap_tbl, "spearman": spearman_tbl}, axis=1).loc[order]
summary

top5_overlap          spearman                     
hub                          L  M  S  N        L      M      S      N
scenario                                                             
growth +0.10                 5  5  5  4    0.973  0.991  0.985  0.982
growth -0.10                 4  4  3  5    0.971  0.984  0.981  0.983
seat_demand +0.10            4  3  3  5    0.975  0.982  0.984  0.989
seat_demand -0.10            5  5  5  3    0.985  0.992  0.987  0.988
congestion +0.10             4  4  4  5    0.978  0.981  0.987  0.988
congestion -0.10             5  4  4  4    0.977  0.988  0.988  0.989
scale +0.10                  5  4  5  5    0.975  0.979  0.979  0.990
scale -0.10                  4  4  4  4    0.981  0.970  0.986  0.991

In [17]:
print("Lowest top-5 overlap:", sens["top5_overlap"].min(), "of 5")
print("Lowest Spearman:     ", round(sens["spearman"].min(), 3))

Lowest top-5 overlap: 3 of 5
Lowest Spearman:      0.97


### Step 6: ranks of the spot-check airports in each scenario

`rank_in_hub` per scenario. Stable numbers across a row mean the airport's position does not depend on the exact weights.

In [18]:
SPOT = ["BOS", "SFO", "LAX", "SNA", "ANC", "HVN"]
spot_ranks = pd.DataFrame({name: res.loc[SPOT, "rank_in_hub"] for name, res in results.items()}).T
spot_ranks.columns = [f"{c} ({base.loc[c, 'hub']}, of {base.loc[c, 'hub_size_n']})" for c in SPOT]
spot_ranks

,"BOS (L, of 30)","SFO (L, of 30)","LAX (L, of 30)","SNA (M, of 35)","ANC (M, of 35)","HVN (S, of 79)"
baseline,9,7,21,9,22,29
growth +0.10,10,8,25,9,21,23
growth -0.10,8,7,15,7,22,42
seat_demand +0.10,9,6,15,6,23,37
seat_demand -0.10,9,6,24,9,20,23
congestion +0.10,8,7,23,10,17,32
congestion -0.10,10,7,21,5,24,29
scale +0.10,9,6,17,5,26,37
scale -0.10,9,6,25,11,19,27


### Step 7: who enters or leaves the top 5?

For each scenario and hub, the airports that join (`in`) or drop out of (`out`) the baseline top 5. Only changes are listed.

In [19]:
changes = []
for name, res in results.items():
    if name == "baseline":
        continue
    for hub in HUBS:
        b, s_ = set(top5(base, hub)), set(top5(res, hub))
        if b != s_:
            changes.append({"scenario": name, "hub": hub,
                            "in": ", ".join(sorted(s_ - b)),
                            "out": ", ".join(sorted(b - s_))})

pd.DataFrame(changes) if changes else "No changes to any top 5"

,scenario,hub,in,out
0,growth +0.10,N,BGR,ROA
1,growth -0.10,L,EWR,ORD
2,growth -0.10,M,JAX,CHS
3,growth -0.10,S,"IWA, SGF","BZN, SRQ"
4,seat_demand +0.10,L,EWR,ORD
5,seat_demand +0.10,M,"JAX, RIC","CHS, RDU"
6,seat_demand +0.10,S,"IWA, SGF","BZN, SRQ"
7,seat_demand -0.10,N,"EGE, VRB","RAP, ROA"
8,congestion +0.10,L,EWR,ORD
9,congestion +0.10,M,JAX,RDU


## Part 4 – Baseline comparison

Does our 4-component score add anything over a simple rule? We compare it with two naive rankings an analyst might use without a model:
- **size_only:** biggest airport first (`enpl_2025`)
- **growth_only:** fastest-growing first (`cagr_19_25`)

If our ranking were almost identical to one of them, the extra components would add little. If it differs, we should be able to explain *why* from the components.

### Step 1: score and build the two naive rankings

Ranked airports only (D18), ranked within their hub group, like our score. 4 ranked airports have no 2019 data, so no CAGR: they go last in growth_only.

In [20]:
from app.core.scoring import compute_scores, load_metrics

p4 = compute_scores(load_metrics())
p4 = p4[p4["ranked"]].copy()
by_hub = p4.groupby("hub")

p4["rank_ours"] = p4["rank_in_hub"].astype(int)
p4["rank_size"] = by_hub["enpl_2025"].rank(ascending=False, method="min").astype("Int64")
p4["rank_growth"] = by_hub["cagr_19_25"].rank(ascending=False, method="min").astype("Int64")

HUBS = ["L", "M", "S", "N"]
HUB_NAMES = {"L": "Large", "M": "Medium", "S": "Small", "N": "Nonhub"}
print("Ranked airports:", len(p4), "| without CAGR:", p4["cagr_19_25"].isna().sum())

Ranked airports: 395 | without CAGR: 4


### Step 2: top 5 side by side

For each hub group, the top 5 under our score, size_only and growth_only. Codes that appear in more than one column are where the rankings agree.

In [21]:
def top5_list(hub, rank_col):
    d = p4[p4["hub"] == hub].sort_values([rank_col, "code"])  # NaN ranks go last
    return [f"{r.code} ({r.city})" for r in d.head(5).itertuples()]

for hub in HUBS:
    print(f"=== {HUB_NAMES[hub]} ===")
    side = pd.DataFrame({"our score": top5_list(hub, "rank_ours"),
                         "size_only": top5_list(hub, "rank_size"),
                         "growth_only": top5_list(hub, "rank_growth")},
                        index=range(1, 6))
    display(side)

=== Large ===


,our score,size_only,growth_only
1,DEN (Denver),ATL (Atlanta),BNA (Nashville)
2,MIA (Miami),DFW (Fort Worth),AUS (Austin)
3,DFW (Fort Worth),ORD (Chicago),MIA (Miami)
4,MCO (Orlando),DEN (Denver),DEN (Denver)
5,ORD (Chicago),LAX (Los Angeles),IAD (Dulles)


=== Medium ===


,our score,size_only,growth_only
1,SJU (San Juan),MDW (Chicago),SJU (San Juan)
2,DJT (West Palm Beach),PDX (Portland),CHS (Charleston)
3,RSW (Fort Myers),DAL (Dallas),ONT (Ontario)
4,RDU (Raleigh),RDU (Raleigh),BOI (Boise)
5,CHS (Charleston),STL (St. Louis),ORF (Norfolk)


=== Small ===


,our score,size_only,growth_only
1,TYS (Alcoa),RNO (Reno),HVN (New Haven)
2,PGD (Punta Gorda),MEM (Memphis),PVU (Provo)
3,PIE (Clearwater),OKC (Oklahoma City),SRQ (Sarasota)
4,BZN (Bozeman),SRQ (Sarasota),BZN (Bozeman)
5,SRQ (Sarasota),SDF (Louisville),HRL (Harlingen)


=== Nonhub ===


,our score,size_only,growth_only
1,BLV (Belleville),FWA (Fort Wayne),VRB (Vero Beach)
2,TVC (Traverse City),CAK (Akron),RIW (Riverton)
3,FWA (Fort Wayne),ABE (Allentown),LNS (Lititz)
4,RAP (Rapid City),TVC (Traverse City),JST (Johnstown)
5,ROA (Roanoke),RAP (Rapid City),AOO (Altoona)


### Step 3: Spearman correlation with each naive ranking

How similar is our *full* ranking within each hub to each naive one? 1.0 = same order, 0 = unrelated. Computed as the correlation of the two rank lists (airports without CAGR are left out of the growth comparison).

In [22]:
rows = []
for hub in HUBS:
    d = p4[p4["hub"] == hub]
    ours = d["investment_score"].rank()
    rows.append({"hub": hub, "n": len(d),
                 "spearman_vs_size": ours.corr(d["enpl_2025"].rank()),
                 "spearman_vs_growth": ours.corr(d["cagr_19_25"].rank())})
pd.DataFrame(rows).set_index("hub").round(3)

,n,spearman_vs_size,spearman_vs_growth
hub,,,
L,30,0.572,0.488
M,35,-0.066,0.780
S,79,0.082,0.663
N,251,0.657,0.614


### Step 4: notable disagreements

Airports in the **top 5 of a naive ranking but outside our top 10**, and the reverse: **our top 5 but outside the naive top 10**. Component percentiles and raw metrics are shown so each difference can be explained.

In [23]:
found = []
for hub in HUBS:
    d = p4[p4["hub"] == hub]
    for naive in ["size", "growth"]:
        naive_rank = d[f"rank_{naive}"].astype("float").fillna(float("inf"))  # no CAGR = last
        naive_top = d[(naive_rank <= 5) & (d["rank_ours"] > 10)]
        ours_top = d[(d["rank_ours"] <= 5) & (naive_rank > 10)]
        found.append(naive_top.assign(disagreement=f"{naive}_only top 5, not our top 10"))
        found.append(ours_top.assign(disagreement=f"our top 5, not {naive}_only top 10"))

disagree = pd.concat(found)
cols = ["hub", "code", "city", "disagreement", "rank_ours", "rank_size", "rank_growth", "hub_size_n",
        "growth_pct", "seat_demand_pct", "congestion_pct", "scale_pct",
        "enpl_2025", "cagr_19_25", "pct_change_24_25", "load_factor", "nas_delay_rate"]
disagree["hub"] = pd.Categorical(disagree["hub"], HUBS, ordered=True)
(disagree[cols].sort_values(["hub", "disagreement", "rank_ours"])
 .style.format({"growth_pct": "{:.2f}", "seat_demand_pct": "{:.2f}",
                "congestion_pct": "{:.2f}", "scale_pct": "{:.2f}",
                "enpl_2025": "{:,.0f}", "cagr_19_25": "{:.1%}", "pct_change_24_25": "{:.1%}",
                "load_factor": "{:.1%}", "nas_delay_rate": "{:.1%}"}, na_rep="–")
 .hide(axis="index"))

hub,code,city,disagreement,rank_ours,rank_size,rank_growth,hub_size_n,growth_pct,seat_demand_pct,congestion_pct,scale_pct,enpl_2025,cagr_19_25,pct_change_24_25,load_factor,nas_delay_rate
L,IAD,Dulles,"growth_only top 5, not our top 10",11,22,5,30,0.97,0.47,0.37,0.30,"13,859,569",2.6%,6.6%,80.3%,6.0%
L,BNA,Nashville,"growth_only top 5, not our top 10",15,25,1,30,1.00,0.10,0.33,0.20,"12,549,304",5.8%,4.1%,76.3%,5.9%
L,AUS,Austin,"growth_only top 5, not our top 10",17,29,2,30,0.93,0.13,0.43,0.07,"10,651,727",3.8%,-0.2%,76.6%,6.3%
L,ORD,Chicago,"our top 5, not growth_only top 10",5,3,21,30,0.53,0.70,0.83,0.93,"40,680,735",-0.1%,5.5%,81.6%,9.5%
L,ATL,Atlanta,"size_only top 5, not our top 10",14,1,23,30,0.20,0.97,0.27,1.00,"51,459,786",-0.6%,-2.0%,82.7%,4.8%
L,LAX,Los Angeles,"size_only top 5, not our top 10",21,5,30,30,0.07,0.77,0.23,0.87,"36,497,303",-2.7%,-3.4%,81.7%,4.6%
M,BOI,Boise,"growth_only top 5, not our top 10",11,30,4,35,0.97,0.86,0.20,0.17,"2,579,647",3.8%,4.2%,79.6%,3.4%
M,ONT,Ontario,"growth_only top 5, not our top 10",19,23,3,35,0.86,0.31,0.17,0.37,"3,473,874",4.1%,-0.6%,74.8%,3.2%
M,DJT,West Palm Beach,"our top 5, not size_only top 10",2,21,6,35,0.94,0.77,1.00,0.43,"4,257,185",3.5%,3.1%,79.3%,10.7%
M,RSW,Fort Myers,"our top 5, not size_only top 10",3,12,10,35,0.77,0.74,0.94,0.69,"5,504,420",1.5%,1.5%,79.1%,7.6%


In [24]:
s.loc[s["code"] == "VRB", ["arr_flights", "departures_FL", "nas_delay_rate"]]

,arr_flights,departures_FL,nas_delay_rate
17,42.0,1372.0,0.254524
